# kernel

>  Connecting a solveit kernel to a Modal sandbox

```mermaid
flowchart TD
    A[Start the remote kernel] --> B[Register the remote kernel]
    B --> C[Connect to the remote kernel]
```

In [ ]:
#| default_exp kernel

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| hide 
enable_mermaid()

<script type="module">
if (window.mermaid) mermaid.run()
else {
    import('https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}</script>

In [ ]:
from solveit_modal.sandbox import *

We'll first spin up a sandbox.

In [ ]:
app = mk_app()
img = mk_img()
sb = mk_sandbox(app, img, gpu='T4')
host,port = get_tunnel(sb)
inject_pubkey(sb, get_pubkey()); start_ssh(sb)
ssh = mk_ssh(host, port)
verify_ssh(ssh)

∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... - INFO | 2026-06-11 07:43:25,156


✔ sandbox ready - INFO | 2026-06-11 07:43:25,608


✔ gotten tunnel: r432.modal.host:45521 - INFO | 2026-06-11 07:43:38,955


✔ public key injected - INFO | 2026-06-11 07:43:39,233


✔ started ssh service - INFO | 2026-06-11 07:43:39,889


System: Linux
Hostname: modal
User: root
Kernel: 4.4.0
Architecture: x86_64
OS Type: GNU/Linux
GPU: Tesla T4


Next up is starting the remote kernel.

In [ ]:
#| export
import logging

In [ ]:
#| export
logging.basicConfig(level=logging.INFO, format='%(message)s - %(levelname)s | %(asctime)s')
log = logging.getLogger(__name__)

In [ ]:
#| export
from time import sleep

In [ ]:
#| export
from typing import Callable

In [ ]:
#| export
def start_remote_kernel(
    ssh:Callable,                                                           # SSH caller from mk_ssh
    kernel_path:str='/root/.local/share/jupyter/runtime/kernel-ipyf.json',  # kernel JSON path
):
    "Start remote kernel and wait until ready."
    log.info('∞ starting kernel')
    ssh('mkdir', '-p', '/root/.local/share/jupyter/runtime')
    ssh(f'nohup python3 -m ipykernel_launcher -f {kernel_path} >/tmp/kernel.log 2>&1 &')
    for i in range(10):
        if 'ok' in ssh(f'test -f {kernel_path} && echo ok || echo no'): break
        sleep(0.2)
    klog = ssh('cat /tmp/kernel.log')
    assert 'To connect another client' in klog, f'kernel failed to start:\n{klog}'
    log.info(f'✔ remote kernel ready: {kernel_path}')

In [ ]:
start_remote_kernel(ssh)

∞ starting kernel - INFO | 2026-06-11 07:46:39,399


✔ remote kernel ready: /root/.local/share/jupyter/runtime/kernel-ipyf.json - INFO | 2026-06-11 07:46:43,360


In [ ]:
#| export
def get_remote_python(
    ssh:Callable,   # SSH caller from mk_ssh
) -> str:           # path to remote python
    "Return the path of Python on the remote machine."
    return ssh('which python3')

In [ ]:
p = get_remote_python(ssh); p

'/usr/local/bin/python3'

Now that the an IPython kernel is up and running on the Modal sandbox, it's time to register that kernel to SolveIt, and then connect to it.

We'll be doing that with [Dr. Scott Hawley's](https://www.scotthawley.com/) awesome [ipyfernel](https://drscotthawley.github.io/ipyfernel/) library.

In [ ]:
#| export
from ipyfernel.core import *

In [ ]:
#| export
_all_ = ['connect_existing_kernel', 'gip', 'ipf_exec', 'ipf_shutdown', 'ipf_startup',
         'local', 'register_remote_kernel', 'remote', 'set_ssh_config',
         'set_sticky', 'start_remote', 'stop_remote', 'unset_sticky']

In [ ]:
register_remote_kernel(remote_python=p)
connect_existing_kernel(f'{host}:{port}')

ipyf_remote_kernel is already a registered kernel
/app/data/.ssh/config file updated.


Successfully created connection file and forwarded ports!


Success: connected to remote kernel via r432.modal.host:45521


And we're done! Use the `%%remote` cell magic to run code on Modal's sandbox.

In [ ]:
%%remote
import platform
platform.uname()

uname_result(system='Linux', node='modal', release='4.4.0', version='#1 SMP Sun Jan 10 15:06:54 PST 2016', machine='x86_64')


In [ ]:
%%remote
!nvidia-smi

Thu Jun 11 07:56:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       On  |   00000000:19:00.0 Off |                    0 |
| N/A   36C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+------------------------------------------------------------------------------

Alternatively, ipyfernel also has a sticky mode allowing us to run code remotely by default.

In [ ]:
set_sticky()

Code cells will now execute remotely.


In [ ]:
import torch

True


In [ ]:
torch.cuda.is_available()

`%%local` to run code locally.

In [ ]:
%local unset_sticky()

Code cells will now run locally.


In [ ]:
stop_remote()

Code cells will now run locally.
Shutting down remote kernel


In [ ]:
sb.terminate()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()